# Adaptive Decision-Focused Exploration via Posterior Disagreement

**Core idea:** Use the decision disagreement rate δ_t as a per-context exploration-exploitation
switch for the DFE mechanism, not the imputation weight.

**Mechanism:**
1. Draw M posterior-predictive samples: `c̃_i^(m) = μ_i + σ_i · z_i^(m)`, z ~ N(0,1)
2. Solve each: `w^(m) = w*(c̃^(m))`
3. Compute: `δ_t = 1 - (1/C(M,2)) Σ_{m<m'} 1[w^(m) = w^(m')]`
4. Set: `γ_t = γ_0 · δ_t` (exploration intensity)

**DFE with adaptive γ_t (minimization):**
- On-solution arms: `b_i = 1 - γ_t` (reduce LCB → look expensive → depart)
- Contested off-solution: `b_i = 1 + α·φ_i` (boost LCB → look cheap → explore)
- Non-discriminating: `b_i = 1` (standard UCB)
- `values = means - |β| · σ · b`

**Why δ_t should control exploration, not imputation:**
- δ_t high → unfamiliar context, posterior disagrees → explore aggressively, force path
  diversity to fill precision matrix in new directions
- δ_t low → familiar context, posterior agrees → exploit, stick with known-good path
- Adaptive λ for imputation scales a near-zero signal (DI ≈ flat on SP)
- Adaptive γ for exploration scales the thing that actually helps (-5.5% on SP)

**Cost:** M = 5–10 extra solver calls per round (cheap)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import jax
import jax.numpy as jnp
import optax
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

from src.envs.shortest_path import ShortestPathEdgeEnv
from src.envs.synthetic import SyntheticKnapsackEnv
from src.agents.base import run_experiment
from src.solver import ShortestPathSolver, KnapsackSolver
from src.agents.bayesian_utils import (
    init_per_arm_regression,
    compute_ucb_values,
    compute_bayesian_predictions,
    update_bayesian_stats,
)
from src.agents.shared_context.bayesian_spo_di import BayesianSPO_DI_Agent

## Loss Gradient Functions

In [2]:
def spo_grad(c_pred, c_true, solver, is_maximize=True):
    cp = np.asarray(c_pred, dtype=np.float64)
    ct = np.asarray(c_true, dtype=np.float64)
    n = cp.shape[0]
    w_true = solver(ct)
    w_spo = solver(2 * cp - ct)
    if is_maximize:
        return jnp.array(2.0 * (w_spo - w_true) / n, dtype=jnp.float32)
    else:
        return jnp.array(2.0 * (w_true - w_spo) / n, dtype=jnp.float32)


def mse_grad(c_pred, c_true):
    n = c_pred.shape[0]
    return 2.0 * (c_pred - c_true) / n


def make_spo_grad(solver, is_maximize=True):
    return lambda c_pred, c_true: spo_grad(c_pred, c_true, solver, is_maximize)


def make_poly_decay_schedule(init_lr, power=0.25):
    def schedule(count):
        # return init_lr / jnp.power(count + 1, power)
        return init_lr / jnp.sqrt(count+1)
    return schedule

## JIT Kernels

In [3]:
@jax.jit
def _fused_ucb(X, W1, b1, A_inv, b_bayes, beta):
    h = X @ W1 + b1
    theta = jnp.einsum('nij,nj->ni', A_inv, b_bayes)
    means = theta @ h
    Ax = jnp.einsum('nij,j->ni', A_inv, h)
    stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))
    return means + beta * stds

@jax.jit
def _fused_predict(X, W1, b1, A_inv, b_bayes):
    h = X @ W1 + b1
    theta = jnp.einsum('nij,nj->ni', A_inv, b_bayes)
    return theta @ h

@jax.jit
def _fused_bayesian_core(X, W1, b1, A, b_bayes, A_inv, rewards, mask):
    h = X @ W1 + b1
    A_new, b_new, A_inv_new = update_bayesian_stats(
        A, b_bayes, A_inv, h, rewards, mask)
    theta = jnp.einsum('nij,nj->ni', A_inv_new, b_new)
    pred = theta @ h
    return h, pred, theta, A_new, b_new, A_inv_new

@jax.jit
def _bayesian_grads(X, theta, grad_pred):
    grad_h = jnp.einsum('ni,n->i', theta, grad_pred)
    return {'W1': jnp.outer(X, grad_h), 'b1': grad_h}

def _make_step_fn(optimizer):
    @jax.jit
    def step(grads, opt_state, params):
        updates, new_state = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_state
    return step

## Decision Disagreement Rate

The core new function: compute δ_t from posterior samples.

In [4]:
def compute_decision_disagreement(means_np, stds_np, solver, rng, M=10):
    """Compute decision disagreement rate δ_t.
    
    Draw M posterior-predictive samples, solve each, count pairwise disagreements.
    
    Args:
        means_np: (n_arms,) posterior means
        stds_np: (n_arms,) posterior stds
        solver: combinatorial solver
        rng: numpy random generator
        M: number of posterior samples
    
    Returns:
        delta_t: float in [0, 1], decision disagreement rate
        solutions: list of M solution vectors (for diagnostics)
    """
    n_arms = len(means_np)
    solutions = []
    
    for m in range(M):
        z = rng.standard_normal(n_arms)
        c_sample = means_np + stds_np * z
        w_m = solver(c_sample)
        solutions.append((w_m > 0.5).astype(np.int8))
    
    # Count pairwise agreements
    n_pairs = 0
    n_agree = 0
    for i in range(M):
        for j in range(i + 1, M):
            n_pairs += 1
            if np.array_equal(solutions[i], solutions[j]):
                n_agree += 1
    
    delta_t = 1.0 - n_agree / max(n_pairs, 1)
    return delta_t, solutions


def compute_propensities(means_np, stds_np, solver, rng, M=10, p_min=0.05):
    """Estimate per-arm selection propensity from posterior samples.
    
    p_i = P(arm i in optimal solution) estimated by solving M posterior samples.
    
    Args:
        means_np: (n_arms,) posterior means
        stds_np: (n_arms,) posterior stds
        solver: combinatorial solver
        rng: numpy random generator
        M: number of posterior samples
        p_min: minimum propensity clamp to avoid explosion
    
    Returns:
        propensities: (n_arms,) array in [p_min, 1.0]
    """
    n_arms = len(means_np)
    counts = np.zeros(n_arms)
    
    for m in range(M):
        z = rng.standard_normal(n_arms)
        c_sample = means_np + stds_np * z
        w_m = solver(c_sample)
        counts += (np.asarray(w_m) > 0.5).astype(float)
    
    propensities = counts / M
    propensities = np.clip(propensities, p_min, 1.0)
    return propensities

## Agent with Adaptive λ

In [5]:
class LinearBayesianAgent:
    """Bayesian decoupled agent with optional adaptive DFE via posterior disagreement.
    
    Imputation modes:
      'model'         — impute unobserved with model prediction (+ optional mask_grad)
      'disc_explore'  — masked SPO+ gradient (zero on unobserved)
      'disc_impute'   — adversarial imputation via alternatives (DI)
      'ips_dr'        — Doubly Robust: no fabricated targets, IPS-weighted observed gradients
    
    DFE (decision_focused_explore=True):
      Asymmetric exploration in act() using phi from slacks.

    Adaptive DFE (adaptive_dfe=True):
      γ_t = γ_0 · δ_t modulates exploration intensity per-context.

    Gated DI (di_gate_threshold > 0):
      Only apply adversarial imputation when δ_t < threshold.
    """

    def __init__(self, context_dim, n_arms, weights, capacity,
                 lr, loss_grad_fn, hidden_dim=32, beta=1.0, lambda_reg=1.0,
                 weight_decay=1e-4, max_grad_norm=5.0, imputation='model',
                 mask_grad=False, minimize=False, solver=None, seed=None,
                 lr_schedule=None, co_solver=None,
                 contested_alpha=1.0, contested_eps=1e-6,
                 disc_K=5, disc_lambda=0.1,
                 n_thompson_samples=20,
                 decision_focused_explore=False,
                 # --- adaptive DFE ---
                 adaptive_dfe=False,
                 dfe_gamma_0=1.0,
                 n_disagreement_samples=10,
                 # --- adaptive lambda (legacy, for comparison) ---
                 adaptive_lambda=False,
                 # --- confidence-gated DI ---
                 di_gate_threshold=0.0,
                 # --- IPS / DR ---
                 ips_M=10,
                 ips_p_min=0.05,
                 ips_clip_max=5.0,
                 **kwargs):
        self.n_arms = n_arms
        self.weights_np = np.asarray(weights, dtype=np.float64)
        self.capacity = capacity
        self.beta = beta
        self.loss_grad_fn = loss_grad_fn
        self.imputation = imputation
        self.mask_grad = mask_grad
        self.minimize = minimize
        self.t = 0
        self.n_thompson_samples = n_thompson_samples
        self.decision_focused_explore = decision_focused_explore

        self.co_solver = co_solver
        self.contested_alpha = contested_alpha
        self.contested_eps = contested_eps
        self.disc_K = disc_K
        self.disc_lambda_0 = disc_lambda
        self.disc_lambda = disc_lambda
        self.contested_history = []
        self.sym_diff_sizes_history = []

        # Adaptive DFE
        self.adaptive_dfe = adaptive_dfe
        self.dfe_gamma_0 = dfe_gamma_0
        self.n_disagreement_samples = n_disagreement_samples
        self.delta_history = []
        self.gamma_history = []

        # Adaptive lambda (legacy)
        self.adaptive_lambda = adaptive_lambda
        self.lambda_history = []

        # Confidence-gated DI
        self.di_gate_threshold = di_gate_threshold
        self.di_gated_history = []

        # IPS / DR
        self.ips_M = ips_M
        self.ips_p_min = ips_p_min
        self.ips_clip_max = ips_clip_max
        self.propensity_history = []  # track mean propensity per round

        if imputation in ('disc_explore', 'disc_impute') and co_solver is None:
            raise ValueError('co_solver required for disc_explore/disc_impute')
        if decision_focused_explore and co_solver is None:
            raise ValueError('co_solver required for decision_focused_explore')

        if solver is None:
            raise ValueError('solver must be provided')
        self.solver = solver

        key = jax.random.PRNGKey(seed if seed is not None else 0)
        key, init_key = jax.random.split(key)
        self.rng_key = key
        self.rng_np = np.random.default_rng(seed if seed is not None else 0)
        self.W1 = jax.random.normal(init_key, (context_dim, hidden_dim)) * 0.1
        self.b1 = jnp.zeros(hidden_dim)

        lr_or_schedule = lr_schedule if lr_schedule is not None else lr
        self.optimizer = optax.chain(
            optax.clip_by_global_norm(max_grad_norm),
            optax.adamw(lr_or_schedule, weight_decay=weight_decay),
        )
        self.opt_state = self.optimizer.init({'W1': self.W1, 'b1': self.b1})
        self._step = _make_step_fn(self.optimizer)

        self.A, self.b_bayes, self.A_inv = init_per_arm_regression(
            n_arms, hidden_dim, lambda_reg)

    def predict_mean(self, X):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        return np.asarray(_fused_predict(X_j, self.W1, self.b1,
                                         self.A_inv, self.b_bayes))

    def _compute_stds(self, h):
        Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
        return jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))

    def _compute_delta_t(self, means_np, stds_np):
        """Compute δ_t from posterior samples."""
        delta_t, _ = compute_decision_disagreement(
            means_np, stds_np, self.solver, self.rng_np,
            M=self.n_disagreement_samples)
        return delta_t

    def act(self, X):
        self.t += 1
        X_j = jnp.asarray(X, dtype=jnp.float32)

        # Compute features, means, stds for ALL paths (avoid JIT non-determinism)
        h = X_j @ self.W1 + self.b1
        theta = jnp.einsum('nij,nj->ni', self.A_inv, self.b_bayes)
        means = theta @ h
        stds = self._compute_stds(h)

        # --- Decision-focused exploration (DFE) ---
        if self.decision_focused_explore:
            means_np = np.asarray(means, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)

            if self.minimize:
                slacks = self.co_solver.compute_slacks(means_np)
                phi = np.exp(-slacks / (stds_np + self.contested_eps))

                if self.adaptive_dfe:
                    delta_t = self._compute_delta_t(means_np, stds_np)
                    gamma_t = self.dfe_gamma_0 * delta_t
                    self.delta_history.append(delta_t)
                    self.gamma_history.append(gamma_t)
                else:
                    gamma_t = self.dfe_gamma_0

                on_solution = slacks < 1e-8
                phi[on_solution] = 1.0 - gamma_t

                phi_j = jnp.array(phi, dtype=jnp.float32)
                values = means - jnp.abs(self.beta) * stds * phi_j
            else:
                # Maximization: additive bonus
                opt_sol, alts = self.co_solver.extract_k_alternatives(
                    means_np, self.disc_K)
                bonus = np.ones(self.n_arms)
                if alts:
                    disc_edges = np.zeros(self.n_arms, dtype=bool)
                    for _, _, sym_diff in alts:
                        disc_edges |= sym_diff
                    on_solution = opt_sol > 0.5
                    disc_off_sol = disc_edges & (~on_solution)
                    if disc_off_sol.any():
                        slacks = self.co_solver.compute_slacks(means_np)
                        bonus[disc_off_sol] = 1.0 + self.contested_alpha * np.exp(
                            -slacks[disc_off_sol] / (stds_np[disc_off_sol] + self.contested_eps))
                bonus_j = jnp.array(bonus, dtype=jnp.float32)
                values = means + self.beta * stds * bonus_j

            return self.solver(np.asarray(values))

        # --- DI/DX targeted exploration (additive bonus, both min/max) ---
        if self.imputation in ('disc_explore', 'disc_impute'):
            means_np = np.asarray(means, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)

            opt_sol, alts = self.co_solver.extract_k_alternatives(
                means_np, self.disc_K)

            bonus = np.ones(self.n_arms)
            if alts:
                disc_edges = np.zeros(self.n_arms, dtype=bool)
                for _, _, sym_diff in alts:
                    disc_edges |= sym_diff
                on_solution = opt_sol > 0.5
                disc_off_sol = disc_edges & (~on_solution)
                if disc_off_sol.any():
                    slacks = self.co_solver.compute_slacks(means_np)
                    bonus[disc_off_sol] = 1.0 + self.contested_alpha * np.exp(
                        -slacks[disc_off_sol] / (stds_np[disc_off_sol] + self.contested_eps))

            bonus_j = jnp.array(bonus, dtype=jnp.float32)
            if self.minimize:
                values = means - jnp.abs(self.beta) * stds * bonus_j
            else:
                values = means + self.beta * stds * bonus_j
            return self.solver(np.asarray(values))

        # --- Standard UCB (non-JIT, matches DFE/DI computation path) ---
        values = means + self.beta * stds
        return self.solver(np.asarray(values))

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        sel_j = jnp.asarray(selection, dtype=jnp.float32)
        rew_j = jnp.asarray(rewards, dtype=jnp.float32)
        sel_mask = sel_j > 0.5
        obs_mask = jnp.ones(self.n_arms, dtype=jnp.bool_) if full_feedback else sel_mask

        h, pred, theta, self.A, self.b_bayes, self.A_inv = \
            _fused_bayesian_core(X_j, self.W1, self.b1,
                self.A, self.b_bayes, self.A_inv, rew_j, obs_mask)

        # Compute stds once for branches that need it
        stds = None
        if self.imputation in ('disc_explore', 'disc_impute', 'ips_dr') or self.adaptive_lambda:
            stds = self._compute_stds(h)

        # --- Adaptive λ for imputation (legacy, for comparison) ---
        if self.adaptive_lambda and self.imputation == 'disc_impute':
            means_np = np.asarray(pred, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)
            delta_t = self._compute_delta_t(means_np, stds_np)
            self.disc_lambda = self.disc_lambda_0 * delta_t
            self.lambda_history.append(self.disc_lambda)

        # --- Confidence-gated DI ---
        if self.imputation == 'disc_impute' and self.di_gate_threshold > 0:
            if stds is None:
                stds = self._compute_stds(h)
            means_np = np.asarray(pred, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)
            delta_t = self._compute_delta_t(means_np, stds_np)
            self.delta_history.append(delta_t)

            if delta_t < self.di_gate_threshold:
                grad_pred = self._grad_disc_impute(pred, rew_j, obs_mask, stds)
                self.di_gated_history.append(1)
            else:
                grad_pred = self._grad_disc_explore(pred, rew_j, obs_mask)
                self.di_gated_history.append(0)

        elif self.imputation == 'ips_dr':
            grad_pred = self._grad_ips_dr(pred, rew_j, obs_mask, stds)

        elif self.imputation == 'disc_explore':
            grad_pred = self._grad_disc_explore(pred, rew_j, obs_mask)
        elif self.imputation == 'disc_impute':
            grad_pred = self._grad_disc_impute(pred, rew_j, obs_mask, stds)
        else:
            Y_target = jnp.where(obs_mask, rew_j, pred)
            grad_pred = self.loss_grad_fn(pred, Y_target)
            if self.mask_grad:
                grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

        grads = _bayesian_grads(X_j, theta, grad_pred)
        params = {'W1': self.W1, 'b1': self.b1}
        new_params, self.opt_state = self._step(grads, self.opt_state, params)
        self.W1 = jnp.clip(new_params['W1'], -10, 10)
        self.b1 = new_params['b1']

    def _grad_disc_explore(self, pred, rew_j, obs_mask):
        Y_target = jnp.where(obs_mask, rew_j, pred)
        grad_pred = self.loss_grad_fn(pred, Y_target)
        return jnp.where(obs_mask, grad_pred, 0.0)

    def _grad_disc_impute(self, pred, rew_j, obs_mask, stds):
        pred_np = np.asarray(pred, dtype=np.float64)
        stds_np = np.asarray(stds, dtype=np.float64)

        opt_sol, alts = self.co_solver.extract_k_alternatives(
            pred_np, self.disc_K)

        if not alts:
            self.sym_diff_sizes_history.append([])
            return self._grad_disc_explore(pred, rew_j, obs_mask)

        grad_pred = jnp.zeros(self.n_arms)
        all_disc = np.zeros(self.n_arms, dtype=bool)
        round_sym_sizes = []

        for alt_sol, _, sym_diff in alts:
            round_sym_sizes.append(int(sym_diff.sum()))

            imputed = pred_np.copy()
            opt_only = (opt_sol > 0.5) & (alt_sol < 0.5)
            alt_only = (alt_sol > 0.5) & (opt_sol < 0.5)

            if self.minimize:
                imputed[opt_only] += self.contested_alpha * stds_np[opt_only]
                imputed[alt_only] -= self.contested_alpha * stds_np[alt_only]
            else:
                imputed[opt_only] -= self.contested_alpha * stds_np[opt_only]
                imputed[alt_only] += self.contested_alpha * stds_np[alt_only]

            Y_k = np.where(np.asarray(obs_mask), np.asarray(rew_j), imputed)
            grad_k = self.loss_grad_fn(pred, jnp.array(Y_k, dtype=jnp.float32))

            disc_k = jnp.array(sym_diff)
            grad_k = jnp.where(
                obs_mask, grad_k,
                jnp.where(disc_k, self.disc_lambda * grad_k, 0.0))

            grad_pred = grad_pred + grad_k
            all_disc |= sym_diff

        self.contested_history.append(int(all_disc.sum()))
        self.sym_diff_sizes_history.append(round_sym_sizes)
        return grad_pred / len(alts)

    def _grad_ips_dr(self, pred, rew_j, obs_mask, stds):
        """Doubly Robust gradient: IPS-weighted observed + model baseline.
        
        DR target: Y_DR_i = μ̂_i + sel_i/p_i · (Y_i - μ̂_i)
        
        For observed arms: Y_DR_i = μ̂_i + (1/p_i)(Y_i - μ̂_i)
          → upweights rare observations (contested arms with low p_i)
        For unobserved arms: Y_DR_i = μ̂_i (model prediction, zero SPO+ gradient)
          → no fabricated targets, no wrong-neighborhood risk
        
        Properties:
          - Unbiased estimator of full-feedback gradient (under mild conditions)
          - When model is correct: reduces to masked gradient (low variance)
          - When model is wrong: IPS correction saves you (unbiased)
          - Naturally upweights informative (contested) arms
        """
        means_np = np.asarray(pred, dtype=np.float64)
        stds_np = np.asarray(stds, dtype=np.float64)

        # Estimate per-arm propensities from posterior samples
        propensities = compute_propensities(
            means_np, stds_np, self.solver, self.rng_np,
            M=self.ips_M, p_min=self.ips_p_min)

        # Track diagnostics
        obs_np = np.asarray(obs_mask)
        if obs_np.any():
            self.propensity_history.append({
                'mean_p_obs': float(propensities[obs_np].mean()),
                'min_p_obs': float(propensities[obs_np].min()),
                'mean_p_all': float(propensities.mean()),
                'n_low_p': int((propensities < 0.2).sum()),
            })

        # Compute IPS weights: w_i = 1/p_i for observed, 0 for unobserved
        # Clip max weight to avoid variance explosion
        ips_weights = np.where(obs_np, 1.0 / propensities, 0.0)
        ips_weights = np.clip(ips_weights, 0.0, self.ips_clip_max)

        # Self-normalize: scale so observed weights sum to n_observed
        n_obs = obs_np.sum()
        if n_obs > 0:
            weight_sum = ips_weights[obs_np].sum()
            if weight_sum > 0:
                ips_weights[obs_np] *= n_obs / weight_sum

        # DR target: Y_DR = pred + ips_weight * (reward - pred)
        # For observed: Y_DR_i = pred_i + w_i * (reward_i - pred_i)
        # For unobserved: Y_DR_i = pred_i (ips_weight = 0)
        ips_w_j = jnp.array(ips_weights, dtype=jnp.float32)
        Y_dr = pred + ips_w_j * (rew_j - pred)

        # Compute SPO+ gradient with DR target
        grad_pred = self.loss_grad_fn(pred, Y_dr)

        # Mask gradient: only observed arms contribute
        # (unobserved have Y_dr = pred, so SPO+ gradient from them is near-zero,
        #  but mask explicitly to be safe)
        grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

        return grad_pred

## Experiment Configuration

In [6]:
GRID_SIZE = 8

DATASETS = {
    'sp_nw025': dict(
        env_cls=ShortestPathEdgeEnv,
        env_kwargs=dict(grid_size=GRID_SIZE, context_dim=5, noise_width=0.25),
        minimize=True,
        group='shortest_path',
        n_rounds=2000,
        co_solver_cls=lambda env: ShortestPathSolver(GRID_SIZE),
        # nonstationarity='drift',
        # drift_rate=0.1
    ),
    'ks_nw025': dict(
        env_cls=SyntheticKnapsackEnv,
        env_kwargs=dict(n_items=32, context_dim=5, capacity=20.0, noise_width=0.25),
        minimize=False,
        group='knapsack',
        n_rounds=2000,
        co_solver_cls=lambda env: KnapsackSolver(env.weights, env.capacity),
    ),
    # 'ks_nw025_64': dict(
    #     env_cls=SyntheticKnapsackEnv,
    #     env_kwargs=dict(n_items=64, context_dim=5, capacity=20.0, noise_width=0.25),
    #     minimize=False,
    #     group='knapsack',
    #     n_rounds=1000,
    #     co_solver_cls=lambda env: KnapsackSolver(env.weights, env.capacity),
    # ),

}

HP = dict(lr=0.001, hidden_dim=32, lambda_reg=1.0, beta=1.0)
SEEDS = [0, 1, 2, 5, 6]
EVAL_INTERVAL = 100

## Agent Factory

Hypothesis: exploration (DFE) is the active ingredient, imputation (DI) is not.

Test:
- `BayesianMSE` — MSE baseline
- `BayesianSPO` — masked SPO+ (no imputation, no DFE)
- `SPO+DFE` — masked SPO+ with DFE exploration (no imputation)
- `SPO+DFE-Adaptive` — masked SPO+ with adaptive DFE (no imputation)
- `DI` — DI imputation only (no DFE)
- `DI+DFE` — DI + fixed DFE
- `DI+DFE-Adaptive` — DI + adaptive DFE

If SPO+DFE ≥ DI+DFE on SP, imputation adds nothing. The contribution is exploration.

In [ ]:
def make_agents(env, seed, minimize, co_solver):
    solver = env.solve
    bayes_beta = -HP['beta'] if minimize else HP['beta']
    spo = make_spo_grad(solver, is_maximize=not minimize)
    sched = make_poly_decay_schedule(HP['lr'])

    common = dict(context_dim=env.context_dim, n_arms=env.n_arms,
                  weights=env.weights, capacity=env.capacity)
    bayes = dict(hidden_dim=HP['hidden_dim'], lr=HP['lr'], seed=seed,
                 beta=bayes_beta, lambda_reg=HP['lambda_reg'], solver=solver,
                 minimize=minimize)
    di_common = dict(co_solver=co_solver, contested_alpha=0.5, disc_K=5,
                     disc_lambda=0.1)

    # Derived IPS parameters from problem structure.
    # Under uniform random feasible policy, expected selection size k = capacity / mean(weights).
    # Min observation probability ≈ k/n → p_min = k/n, clip_max = 1/p_min = n/k.
    n = env.n_arms
    k_est = env.capacity / np.mean(env.weights)
    ips_p_min = k_est / n
    ips_clip = n / k_est   # = 1 / p_min
    print(f'  DR: k_est={k_est:.1f}, p_min=k/n={ips_p_min:.3f}, clip_max=n/k={ips_clip:.1f}')

    # All DFE uses adaptive γ_t = γ_0 · δ_t (no γ_0 to tune).
    dfe_common = dict(
        decision_focused_explore=True, dfe_gamma_0=1.0,
        adaptive_dfe=True, n_disagreement_samples=10,
        co_solver=co_solver,
    )

    # Codebase agent common kwargs (uses init_lr, has its own inv-sqrt schedule)
    codebase_common = dict(
        context_dim=env.context_dim, n_arms=env.n_arms,
        weights=env.weights, capacity=env.capacity,
        hidden_dim=HP['hidden_dim'], init_lr=HP['lr'], seed=seed,
        beta=bayes_beta, lambda_reg=HP['lambda_reg'], solver=solver,
        minimize=minimize, co_solver=co_solver,
        contested_alpha=0.5, disc_K=5, disc_lambda=0.1,
    )

    return {
        'BayesianMSE': LinearBayesianAgent(
            **common, **bayes, loss_grad_fn=mse_grad,
            imputation='model', lr_schedule=sched),

        'BayesianSPO': LinearBayesianAgent(
            **common, **bayes, loss_grad_fn=spo,
            imputation='model', mask_grad=True, lr_schedule=sched),

        # --- DFE (adaptive γ_t = γ_0 · δ_t) ---
        'DFE': LinearBayesianAgent(
            **common, **bayes, loss_grad_fn=spo,
            imputation='model', mask_grad=True,
            **dfe_common, lr_schedule=sched),

        # --- DFE + DR gradient ---
        'DFE+DR': LinearBayesianAgent(
            **common, **bayes, loss_grad_fn=spo,
            imputation='ips_dr', ips_M=10, ips_p_min=ips_p_min, ips_clip_max=ips_clip,
            **dfe_common, lr_schedule=sched),

        # --- DI only (no DFE) ---
        'DI': LinearBayesianAgent(
            **common, **bayes, loss_grad_fn=spo,
            imputation='disc_impute', **di_common,
            lr_schedule=sched),

        # --- Doubly Robust (IPS-weighted observed, clip = n/k) ---
        'DR': LinearBayesianAgent(
            **common, **bayes, loss_grad_fn=spo,
            imputation='ips_dr', ips_M=10, ips_p_min=ips_p_min, ips_clip_max=ips_clip,
            lr_schedule=sched),

        # --- Codebase agents (BayesianSPO_DI_Agent, inv-sqrt LR schedule) ---
        'DX (codebase)': BayesianSPO_DI_Agent(
            **codebase_common, imputation='disc_explore'),

        'DI (codebase)': BayesianSPO_DI_Agent(
            **codebase_common, imputation='disc_impute'),
    }

## Pre-generate Episode Data for Agent Isolation

**Problem:** Adding/removing agents from the experiment dictionary changes results for other
agents, even though each gets its own `env_run`. Something (likely Gurobi internal state or
a hidden global RNG) leaks between sequential agent runs.

**Fix:** Pre-generate all (context, true_values) sequences and test sets per (dataset, seed)
BEFORE running any agents. Every agent replays the exact same data sequence, making results
independent of what other agents are present.

In [8]:
from src.agents.base import _compute_test_regret
import src.solver as _solver_module


def pregenerate_episode(env_cls, env_kwargs, seed, n_rounds, n_test_samples=100):
    """Pre-generate all data for a (dataset, seed) pair.
    
    Creates one env, runs it through the exact same sequence as run_experiment
    (generate_test_set → reset → T rounds of step+get_context), recording
    every (context, true_values, optimal_action, optimal_value).
    
    Returns a dict that can be replayed for any number of agents without
    any cross-agent contamination.
    """
    env = env_cls(**env_kwargs, seed=seed)
    test_set = env.generate_test_set(n_test_samples)
    solver = env.solve
    minimize = getattr(env, 'minimize', False)

    X = env.reset()

    rounds_data = []
    for t in range(n_rounds):
        # Capture current round's data
        if hasattr(env, '_current_costs'):          # ShortestPathEdgeEnv
            true_vals = env._current_costs.copy()
        else:                                       # KS (ConstrainedBanditEnv)
            true_vals = env.current_values.copy()

        optimal = solver(true_vals)
        optimal_val = float(np.sum(true_vals * np.asarray(optimal)))

        rounds_data.append({
            'context': X.copy(),
            'true_values': true_vals,
            'optimal_action': np.asarray(optimal).copy(),
            'optimal_value': optimal_val,
        })

        # Advance env exactly as run_experiment does
        dummy = np.zeros(env.n_arms)
        env.step(dummy, full_feedback=False)
        X = env.get_context()

    return {
        'test_set': test_set,
        'rounds_data': rounds_data,
        'env': env,          # for weights, capacity, solver, minimize
        'solver': solver,
        'minimize': minimize,
    }


def _reset_gurobi_env():
    """Reset the global Gurobi environment so each agent starts fresh.
    
    Gurobi's shared environment accumulates internal state (branching stats,
    tie-breaking history) across model solves. This makes results depend on
    which agents ran previously. Resetting before each agent ensures
    deterministic, order-independent results.
    """
    _solver_module._GUROBI_ENV = None


def run_agent_isolated(agent, episode, n_rounds, lr, eval_interval,
                       full_feedback=False):
    """Run one agent on pre-generated episode data.
    
    Replays the exact (context, values) sequence from `episode`, so results
    are independent of what other agents exist or ran before.
    """
    # Reset Gurobi state so this agent's solves are independent of prior agents
    _reset_gurobi_env()

    test_set = episode['test_set']
    rounds_data = episode['rounds_data']
    env = episode['env']
    solver = episode['solver']
    minimize = episode['minimize']
    X_test, Y_test = test_set

    regrets = []
    cumulative_regret = []
    rewards_list = []
    optimal_rewards_list = []
    test_regrets = []
    actions_list = []
    optimal_actions_list = []
    total_regret = 0.0
    total_reward = 0.0
    total_optimal = 0.0

    for t in range(n_rounds):
        rd = rounds_data[t]
        X = rd['context']
        true_vals = rd['true_values']
        optimal_action = rd['optimal_action']
        optimal_val = rd['optimal_value']

        # Agent acts
        selection = agent.act(X)
        selection = np.asarray(selection).flatten()

        # Compute reward and regret
        agent_val = float(np.sum(true_vals * selection))
        if minimize:
            instant_regret = agent_val - optimal_val
            reward = -agent_val
        else:
            instant_regret = optimal_val - agent_val
            reward = agent_val

        total_regret += instant_regret
        total_reward += reward
        total_optimal += optimal_val

        regrets.append(instant_regret)
        cumulative_regret.append(total_regret)
        rewards_list.append(reward)
        optimal_rewards_list.append(optimal_val)
        actions_list.append(selection.copy())
        optimal_actions_list.append(optimal_action.copy())

        # Observed values (semi-bandit feedback)
        if full_feedback:
            observed = true_vals.copy()
        else:
            observed = np.zeros(len(true_vals))
            mask = selection > 0.5
            observed[mask] = true_vals[mask]

        # Test regret evaluation (same schedule as run_experiment)
        if t == n_rounds - 1 or (t > 0 and t % eval_interval == 0):
            test_regret = _compute_test_regret(
                agent, X_test, Y_test, env.weights, env.capacity,
                solver, minimize)
            test_regrets.append(test_regret)
        else:
            test_regrets.append(np.nan)

        # Agent update
        try:
            agent.update(X, selection, observed, lr, full_feedback=full_feedback)
        except TypeError:
            agent.update(X, selection, observed, full_feedback=full_feedback)

    return {
        'rewards': rewards_list,
        'optimal_rewards': optimal_rewards_list,
        'regrets': regrets,
        'cumulative_regret': cumulative_regret,
        'final_regret': total_regret,
        'final_reward': total_reward,
        'final_optimal': total_optimal,
        'test_regrets': test_regrets,
        'actions': actions_list,
        'optimal_actions': optimal_actions_list,
    }


print('Pre-generation and isolated run functions defined (with Gurobi env reset).')

Pre-generation and isolated run functions defined (with Gurobi env reset).


In [9]:
all_results = {}
all_agents = {}

for ds_name, ds_cfg in DATASETS.items():
    env_cls = ds_cfg['env_cls']
    env_kwargs = ds_cfg['env_kwargs']
    minimize = ds_cfg['minimize']
    n_rounds = ds_cfg['n_rounds']

    print(f'\n=== {ds_name} (minimize={minimize}, T={n_rounds}) ===')
    all_results[ds_name] = {}
    all_agents[ds_name] = {}

    for seed in SEEDS:
        # --- Phase 1: Pre-generate ALL data for this (dataset, seed) ---
        # Reset Gurobi env before pre-generation so optimal actions are
        # computed from a fresh state, independent of prior seeds/datasets.
        _reset_gurobi_env()
        episode = pregenerate_episode(
            env_cls, env_kwargs, seed, n_rounds, n_test_samples=100)
        env = episode['env']

        # Create co_solver and agents from the pre-generated env
        co_solver = ds_cfg['co_solver_cls'](env)
        agents = make_agents(env, seed, minimize, co_solver)

        # --- Phase 2: Run each agent on identical pre-generated data ---
        for agent_name, agent in agents.items():
            if agent_name not in all_results[ds_name]:
                all_results[ds_name][agent_name] = {}
                all_agents[ds_name][agent_name] = {}

            result = run_agent_isolated(
                agent, episode, n_rounds=n_rounds,
                lr=HP['lr'], eval_interval=EVAL_INTERVAL,
                full_feedback=False)

            all_results[ds_name][agent_name][seed] = result
            all_agents[ds_name][agent_name][seed] = agent
            print(f'  {agent_name:<28s} seed={seed}  regret={result["final_regret"]:.1f}')


=== sp_nw025 (minimize=True, T=2000) ===
  DR: k_est=14.0, p_min=k/n=0.125, clip_max=n/k=8.0
  BayesianMSE                  seed=0  regret=7703.7
  BayesianSPO                  seed=0  regret=7205.5
  DFE                          seed=0  regret=6808.4
  DFE+DR                       seed=0  regret=6828.0
  DI                           seed=0  regret=7109.0
  DR                           seed=0  regret=6982.7
  DR: k_est=14.0, p_min=k/n=0.125, clip_max=n/k=8.0
  BayesianMSE                  seed=1  regret=6778.6
  BayesianSPO                  seed=1  regret=6067.7
  DFE                          seed=1  regret=5667.7
  DFE+DR                       seed=1  regret=5610.8
  DI                           seed=1  regret=5911.9
  DR                           seed=1  regret=6080.9
  DR: k_est=14.0, p_min=k/n=0.125, clip_max=n/k=8.0
  BayesianMSE                  seed=2  regret=8166.7
  BayesianSPO                  seed=2  regret=7111.8
  DFE                          seed=2  regret=6696.2
  DFE+D

## Summary Statistics

In [14]:
for ds_name, ds_results in all_results.items():
    print(f'\n{ds_name}')
    print(f'{"Agent":<30s} {"Final Regret":>14s} {"% vs BayesianMSE":>16s}')
    print('-' * 64)

    mse_final = np.mean([ds_results['BayesianMSE'][s]['final_regret'] for s in SEEDS])

    for agent_name in ds_results:
        seed_results = ds_results[agent_name]
        final = np.array([seed_results[s]['final_regret'] for s in SEEDS])
        pct = (final.mean() - mse_final) / mse_final * 100
        sign = '+' if pct >= 0 else ''
        print(f'  {agent_name:<28s} {final.mean():>8.1f} ± {final.std():>5.1f}'
              f'  {sign}{pct:.1f}%')


sp_nw025
Agent                            Final Regret % vs BayesianMSE
----------------------------------------------------------------
  BayesianMSE                    7630.8 ± 742.7  +0.0%
  BayesianSPO                    6880.9 ± 611.0  -9.8%
  DFE                            6563.9 ± 634.4  -14.0%
  DFE+DR                         6524.3 ± 619.0  -14.5%
  DI                             6788.9 ± 639.6  -11.0%
  DR                             6740.4 ± 586.8  -11.7%

ks_nw025
Agent                            Final Regret % vs BayesianMSE
----------------------------------------------------------------
  BayesianMSE                    8362.5 ± 751.2  +0.0%
  BayesianSPO                    7901.9 ± 666.2  -5.5%
  DFE                            8704.3 ± 664.2  +4.1%
  DFE+DR                         9268.5 ± 718.5  +10.8%
  DI                             7870.7 ± 617.2  -5.9%
  DR                             7931.3 ± 694.7  -5.2%


## Plot: Regret Curves

In [ ]:
from lets_plot import gggrid, ggsave

FIGDIR = 'lets-plot-images'
os.makedirs(FIGDIR, exist_ok=True)

COLORS = {
    'BayesianMSE':        '#d62728',
    'BayesianSPO':        '#1f77b4',
    'DFE':                '#9467bd',
    'DFE+DR':             '#bcbd22',
    'DI':                 '#2ca02c',
    'DR':                 '#ff7f0e',
    'DX (codebase)':      '#17becf',
    'DI (codebase)':      '#e377c2',
}

LINETYPES = {
    'BayesianMSE':        'dotted',
    'BayesianSPO':        'dashed',
    'DFE':                'solid',
    'DFE+DR':             'solid',
    'DI':                 'solid',
    'DR':                 'solid',
    'DX (codebase)':      'dashed',
    'DI (codebase)':      'dashed',
}

WINDOW = 50
LINE_LEGEND_THEME = theme(legend_position='bottom', legend_title=element_blank(),
                          legend_text=element_text(size=8))

rows_cum, rows_pr, rows_box = [], [], []
for ds_name, ds_cfg in DATASETS.items():
    ds_results = all_results[ds_name]
    for agent_name, seed_results in ds_results.items():
        for seed in SEEDS:
            result = seed_results[seed]
            for t, r in enumerate(result['cumulative_regret'], 1):
                rows_cum.append(dict(dataset=ds_name, agent=agent_name,
                                     seed=seed, round=t, cumulative_regret=r))
            kernel = np.ones(WINDOW) / WINDOW
            smoothed = np.convolve(result['regrets'], kernel, mode='valid')
            for t, r in enumerate(smoothed):
                rows_pr.append(dict(dataset=ds_name, agent=agent_name,
                                    seed=seed, round=t + WINDOW, per_round_regret=r))
            rows_box.append(dict(
                dataset=ds_name, agent=agent_name, seed=seed,
                final_cumulative_regret=result['final_regret'],
                last_round_regret=np.mean(result['regrets'][-WINDOW:]),
            ))

df_cum = pd.DataFrame(rows_cum)
df_cum_agg = df_cum.groupby(['dataset', 'agent', 'round']).agg(
    mean=('cumulative_regret', 'mean'), std=('cumulative_regret', 'std')
).reset_index()
df_cum_agg['ymin'] = df_cum_agg['mean'] - df_cum_agg['std']
df_cum_agg['ymax'] = df_cum_agg['mean'] + df_cum_agg['std']

df_pr = pd.DataFrame(rows_pr)
df_pr_agg = df_pr.groupby(['dataset', 'agent', 'round']).agg(
    mean=('per_round_regret', 'mean'), std=('per_round_regret', 'std')
).reset_index()
df_pr_agg['ymin'] = df_pr_agg['mean'] - df_pr_agg['std']
df_pr_agg['ymax'] = df_pr_agg['mean'] + df_pr_agg['std']

df_box = pd.DataFrame(rows_box)

all_plots = []
for ds_name in DATASETS:
    d_cum = df_cum_agg[df_cum_agg['dataset'] == ds_name]
    d_pr = df_pr_agg[df_pr_agg['dataset'] == ds_name]
    d_box = df_box[df_box['dataset'] == ds_name].copy()
    label = ds_name

    order = (d_box.groupby('agent')['final_cumulative_regret']
             .median().sort_values().index.tolist())
    d_box['agent'] = pd.Categorical(d_box['agent'], categories=order, ordered=True)

    p_cum = (ggplot(d_cum, aes(x='round', y='mean', color='agent', fill='agent'))
        + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
        + geom_line(aes(linetype='agent'), size=1.0)
        + scale_color_manual(values=COLORS) + scale_fill_manual(values=COLORS)
        + scale_linetype_manual(values=LINETYPES)
        + guides(color=guide_legend(nrow=2), fill='none', linetype=guide_legend(nrow=2))
        + labs(title=f'{label}: Cumulative Regret', x='Round', y='Cumulative Regret')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_pr = (ggplot(d_pr, aes(x='round', y='mean', color='agent', fill='agent'))
        + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
        + geom_line(aes(linetype='agent'), size=1.0)
        + scale_color_manual(values=COLORS) + scale_fill_manual(values=COLORS)
        + scale_linetype_manual(values=LINETYPES)
        + guides(color=guide_legend(nrow=2), fill='none', linetype=guide_legend(nrow=2))
        + labs(title=f'{label}: Per-Round ({WINDOW}-avg)', x='Round', y='Per-Round Regret')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_box = (ggplot(d_box, aes(x='agent', y='final_cumulative_regret', fill='agent'))
        + geom_boxplot(alpha=0.7, width=0.6)
        + geom_jitter(aes(color='agent'), width=0.15, size=2.5, alpha=0.8)
        + scale_fill_manual(values=COLORS) + scale_color_manual(values=COLORS)
        + labs(title=f'{label}: Final Regret', x='', y='Cumulative Regret')
        + theme_minimal()
        + theme(legend_position='none', axis_text_x=element_text(angle=45, hjust=1)))

    all_plots.extend([p_cum, p_pr, p_box])

fig = gggrid(all_plots, ncol=3, align=True, hspace=30, vspace=40) + ggsize(1800, 1000)
fig.show()
ggsave(fig, os.path.join(FIGDIR, 'adaptive_lambda_overview.svg'))
print(f'Saved: {FIGDIR}/adaptive_lambda_overview.svg')

## Plot: δ_t and γ_t Trajectories

Diagnostic: how δ_t (disagreement rate) and γ_t = γ₀ · δ_t (exploration intensity) evolve.
δ_t should decrease as the posterior converges. γ_t controls how aggressively DFE
removes LCB from on-solution arms.

In [12]:
# Extract δ_t trajectories from DFE agents (all DFE is now adaptive)
DELTA_AGENTS = ['DFE', 'DFE+DR']

rows_delta = []
for ds_name in DATASETS:
    for agent_name in DELTA_AGENTS:
        if agent_name not in all_agents.get(ds_name, {}):
            continue
        for seed in SEEDS:
            agent = all_agents[ds_name][agent_name][seed]
            if not hasattr(agent, 'delta_history') or not agent.delta_history:
                continue
            kernel = np.ones(WINDOW) / WINDOW
            delta_smooth = np.convolve(agent.delta_history, kernel, mode='valid')
            for t in range(len(delta_smooth)):
                rows_delta.append(dict(
                    dataset=ds_name, agent=agent_name, seed=seed,
                    round=t + WINDOW,
                    delta_t=delta_smooth[t]))

if rows_delta:
    df_delta = pd.DataFrame(rows_delta)
    df_delta_agg = df_delta.groupby(['dataset', 'agent', 'round']).agg(
        delta_mean=('delta_t', 'mean'),
    ).reset_index()

    delta_plots = []
    for ds_name in DATASETS:
        d = df_delta_agg[df_delta_agg['dataset'] == ds_name]
        if d.empty:
            continue

        p_delta = (ggplot(d, aes(x='round', y='delta_mean', color='agent'))
            + geom_line(size=1.0)
            + scale_color_manual(values=COLORS)
            + labs(title=f'{ds_name}: δ_t (disagreement rate)', x='Round', y='δ_t')
            + theme_minimal() + LINE_LEGEND_THEME)

        delta_plots.append(p_delta)

    if delta_plots:
        fig_delta = gggrid(delta_plots, ncol=1, align=True, hspace=30, vspace=30) + ggsize(900, 400 * len(delta_plots))
        fig_delta.show()
        ggsave(fig_delta, os.path.join(FIGDIR, 'dfe_delta_trajectories.svg'))
        print(f'Saved: {FIGDIR}/dfe_delta_trajectories.svg')
else:
    print('No δ_t data found for DFE agents.')

Saved: lets-plot-images/dfe_delta_trajectories.svg


In [13]:
# DR propensity diagnostics
DR_AGENTS = ['DR', 'DFE+DR']

for ds_name in DATASETS:
    print(f'\n=== {ds_name}: DR propensity statistics ===')
    n_rounds = DATASETS[ds_name]['n_rounds']
    stages = [
        ('Early (1-200)', 0, 200),
        ('Mid (400-600)', 400, 600),
        (f'Late ({n_rounds-200}-{n_rounds})', max(0, n_rounds-200), n_rounds),
    ]

    for agent_name in DR_AGENTS:
        if agent_name not in all_agents.get(ds_name, {}):
            continue
        print(f'\n  {agent_name}:')
        print(f'    {"Stage":<25s} {"mean_p_obs":>12s} {"min_p_obs":>12s} {"n_low_p":>10s}')
        print(f'    {"-"*59}')
        for stage_name, t_start, t_end in stages:
            mean_ps, min_ps, low_ps = [], [], []
            for seed in SEEDS:
                agent = all_agents[ds_name][agent_name].get(seed)
                if agent is None or not agent.propensity_history:
                    continue
                for t in range(t_start, min(t_end, len(agent.propensity_history))):
                    ph = agent.propensity_history[t]
                    mean_ps.append(ph['mean_p_obs'])
                    min_ps.append(ph['min_p_obs'])
                    low_ps.append(ph['n_low_p'])
            if mean_ps:
                print(f'    {stage_name:<25s} {np.mean(mean_ps):>12.3f} {np.mean(min_ps):>12.3f} {np.mean(low_ps):>10.1f}')

        # Per-seed results
        print(f'\n    Per-seed final regret:')
        for seed in SEEDS:
            dr_regret = all_results[ds_name][agent_name][seed]['final_regret']
            spo_regret = all_results[ds_name]['BayesianSPO'][seed]['final_regret']
            pct_vs_spo = (dr_regret - spo_regret) / spo_regret * 100
            print(f'    seed={seed}: {agent_name}={dr_regret:.0f}  SPO={spo_regret:.0f}'
                  f'  vs_SPO={pct_vs_spo:+.1f}%')


=== sp_nw025: DR propensity statistics ===

  DR:
    Stage                       mean_p_obs    min_p_obs    n_low_p
    -----------------------------------------------------------
    Early (1-200)                    0.191        0.125       80.7
    Mid (400-600)                    0.313        0.132       77.4
    Late (1800-2000)                 0.831        0.592       93.5

    Per-seed final regret:
    seed=0: DR=6983  SPO=7206  vs_SPO=-3.1%
    seed=1: DR=6081  SPO=6068  vs_SPO=+0.2%
    seed=2: DR=6930  SPO=7112  vs_SPO=-2.6%
    seed=5: DR=7618  SPO=7721  vs_SPO=-1.3%
    seed=6: DR=6090  SPO=6298  vs_SPO=-3.3%

  DFE+DR:
    Stage                       mean_p_obs    min_p_obs    n_low_p
    -----------------------------------------------------------
    Early (1-200)                    0.191        0.125       80.9
    Mid (400-600)                    0.301        0.130       77.9
    Late (1800-2000)                 0.588        0.293       85.0

    Per-seed final regret